# 从零复现 DETR：集合预测、手写 QKV 与二分匹配

本 Notebook 不调用检测器、`torchvision.models`、`nn.MultiheadAttention` 或 `nn.Transformer`。我们从 Tiny CNN backbone 开始，显式实现二维位置编码、multi-head Q/K/V attention、encoder、decoder、object queries、类别头与归一化 box head；再用穷举法实现小规模最优二分匹配和 set loss。

除 forward 之外，还要验证：预测顺序不影响集合损失、空目标图像只训练 no-object、padding 像素不会泄漏、box 坐标合法、梯度贯通 backbone/queries/heads、推理过滤 no-object，以及模型制品绑定标签和预处理。

数据完全离线合成、固定 seed、CPU 单线程。微型穷举匹配仅用于教学 oracle，生产规模必须换成经过验证的 Hungarian 实现。

## 1. DETR 的计算图与集合语义

```text
image + pixel padding mask
  -> TinyCNN [B,D,H',W']
  -> flatten + 2D position [B,S,D]
  -> manual encoder self-attention
  -> learned object queries + manual decoder self/cross-attention
  -> class logits [B,Q,K+1] + normalized cxcywh boxes [B,Q,4]
  -> one-to-one matching -> set loss
```

$Q$ 个 query 没有固定“第一个物体、第二个物体”语义。匹配在所有 query 与真实目标间选择一对一分配，未匹配 query 的类别为 $\varnothing$（no-object）。因此训练目标是集合，而不是按数组下标对齐。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from copy import deepcopy
from hashlib import sha256
from itertools import permutations
from types import MappingProxyType
import io
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 350728
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. Tiny CNN backbone 与 padding mask

卷积 backbone 把 `[B,1,16,16]` 变成 `[B,D,4,4]`。输入 padding mask 使用 `True=padding`；进入卷积前必须把 padding 像素清零，否则调用方改变无效区域也会改变边界特征。mask 随分辨率下采样时采用 max pooling：一个 feature cell 的感受野只要覆盖 padding，就保守地标为 padding。

真实 DETR 常用预训练 ResNet；这里手写小 backbone，只复现接口和梯度路径。

In [ ]:
class TinyCNNBackbone(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=16):
        super().__init__()
        self.in_channels = int(in_channels)
        self.network = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=False),
            nn.Conv2d(16, hidden_dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim), nn.ReLU(inplace=False),
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=2, padding=1), nn.ReLU(inplace=False),
        )

    def propagate_padding_mask(self, pixel_padding_mask):
        if pixel_padding_mask.ndim != 3 or pixel_padding_mask.dtype != torch.bool:
            raise ValueError("pixel padding mask must be bool [B,H,W]")
        feature_mask = pixel_padding_mask.float().unsqueeze(1)
        for layer in self.network:
            if isinstance(layer, nn.Conv2d):
                feature_mask = F.max_pool2d(feature_mask, kernel_size=layer.kernel_size,
                                            stride=layer.stride, padding=layer.padding,
                                            dilation=layer.dilation)
        return feature_mask.squeeze(1).bool()

    def forward(self, images):
        if images.ndim != 4 or images.shape[1] != self.in_channels:
            raise ValueError("backbone expects configured NCHW images")
        return self.network(images)

backbone_probe = TinyCNNBackbone()
backbone_input = torch.randn(2, 1, 16, 16, requires_grad=True)
backbone_output = backbone_probe(backbone_input)
backbone_output.mean().backward()
assert backbone_output.shape == (2, 16, 2, 2)
assert backbone_input.grad is not None and float(backbone_input.grad.norm()) > 0
assert torch.isfinite(backbone_input.grad).all()

## 3. 二维正弦位置编码

纯 attention 对 token 排列是等变的，展平 feature map 后必须显式注入行列位置。令频率 $\omega_i=10000^{-2i/d}$，分别计算行、列的 sin/cos，再拼成 $D$ 维。`hidden_dim` 必须被 4 整除，因为 x/y 各占一半，其中又各分 sin/cos。

padding 位置的 encoding 置零；attention 中还会再次以 key padding mask 屏蔽其 logits。两道约束解决的问题不同：前者防止位置值进入残差，后者保证 softmax 不把概率分给无效 key。

In [ ]:
class Sine2DPositionEncoding(nn.Module):
    def __init__(self, hidden_dim=16, temperature=10000.0):
        super().__init__()
        if hidden_dim % 4 != 0:
            raise ValueError("hidden_dim must be divisible by four")
        self.hidden_dim = int(hidden_dim)
        self.temperature = float(temperature)

    def forward(self, padding_mask):
        if padding_mask.ndim != 3 or padding_mask.dtype != torch.bool:
            raise ValueError("padding mask must be bool [B,H,W]")
        batch, height, width = padding_mask.shape
        quarter = self.hidden_dim // 4
        frequency = self.temperature ** (-torch.arange(quarter, device=padding_mask.device) / quarter)
        y = torch.arange(height, device=padding_mask.device, dtype=torch.float32)[:, None] * frequency[None, :]
        x = torch.arange(width, device=padding_mask.device, dtype=torch.float32)[:, None] * frequency[None, :]
        y_encoding = torch.cat([y.sin(), y.cos()], dim=-1)[:, None, :].expand(height, width, -1)
        x_encoding = torch.cat([x.sin(), x.cos()], dim=-1)[None, :, :].expand(height, width, -1)
        position = torch.cat([y_encoding, x_encoding], dim=-1)
        position = position.unsqueeze(0).expand(batch, -1, -1, -1).clone()
        return position.masked_fill(padding_mask[..., None], 0.0)

position_encoder = Sine2DPositionEncoding(16)
position_mask = torch.zeros(2, 4, 5, dtype=torch.bool)
position_mask[:, -1, -1] = True
position = position_encoder(position_mask)
assert position.shape == (2, 4, 5, 16)
assert torch.equal(position[:, -1, -1], torch.zeros(2, 16))
assert not torch.equal(position[:, 0, 0], position[:, 0, 1])

## 4. 手写 multi-head QKV attention

对 query $Q\in\mathbb{R}^{T_q\times D}$ 和 memory $K,V\in\mathbb{R}^{T_k\times D}$：

$$A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right),\qquad Z=AV.$$

reshape 后 attention 为 `[B,heads,Tq,Tk]`。mask 在 softmax 前把 padding key 设为负无穷；若某个样本所有 key 都被 mask，应 fail closed，而不是产生整行 NaN。下面的恒等投影 oracle 与显式矩阵乘法对齐缩放因子和 softmax 轴。

In [ ]:
class ManualMultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must divide num_heads")
        self.hidden_dim = int(hidden_dim)
        self.num_heads = int(num_heads)
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, hidden_dim)

    def _split(self, tensor):
        batch, tokens, _ = tensor.shape
        return tensor.reshape(batch, tokens, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, key_padding_mask=None, return_weights=False):
        if not (query.ndim == key.ndim == value.ndim == 3):
            raise ValueError("attention expects [B,T,D]")
        if query.shape[0] != key.shape[0] or key.shape != value.shape:
            raise ValueError("attention batch/key/value mismatch")
        if query.shape[-1] != self.hidden_dim or key.shape[-1] != self.hidden_dim:
            raise ValueError("attention hidden dimension mismatch")
        q, k, v = self._split(self.query(query)), self._split(self.key(key)), self._split(self.value(value))
        scores = (q @ k.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            if key_padding_mask.shape != key.shape[:2] or key_padding_mask.dtype != torch.bool:
                raise ValueError("invalid key padding mask")
            if key_padding_mask.all(dim=1).any():
                raise ValueError("an example cannot mask every key")
            scores = scores.masked_fill(key_padding_mask[:, None, None, :], float("-inf"))
        weights = scores.softmax(dim=-1)
        context = (weights @ v).transpose(1, 2).contiguous().reshape(query.shape[0], query.shape[1], self.hidden_dim)
        result = self.output(context)
        return (result, weights) if return_weights else result

attention_oracle = ManualMultiHeadAttention(4, 1).eval()
with torch.no_grad():
    for linear in (attention_oracle.query, attention_oracle.key,
                   attention_oracle.value, attention_oracle.output):
        linear.weight.copy_(torch.eye(4)); linear.bias.zero_()
oracle_tokens = torch.tensor([[[1., 0., 0., 0.], [0., 2., 0., 0.]]])
oracle_result, oracle_weights = attention_oracle(oracle_tokens, oracle_tokens, oracle_tokens,
                                                  return_weights=True)
expected_weights = ((oracle_tokens @ oracle_tokens.transpose(1, 2)) / math.sqrt(4)).softmax(-1)
expected_result = expected_weights @ oracle_tokens
assert torch.allclose(oracle_weights[:, 0], expected_weights)
assert torch.allclose(oracle_result, expected_result)
assert torch.allclose(oracle_weights.sum(-1), torch.ones(1, 1, 2))

## 5. 显式 Encoder、Decoder 与 object queries

Encoder 让图像 token 相互聚合；Decoder 先让 object queries 自注意，再以 query 为 Q、encoder memory 为 K/V 做 cross-attention。learned query embedding 是“检测槽位”的身份信息，初始 decoder content 为零。

类别头输出 $K+1$ 类，最后一类固定为 no-object。box MLP 末尾 `sigmoid`，得到相对图像大小的 `(cx,cy,w,h)\in(0,1)^4`。这只保证数值范围，不保证 box 边界完全落在图像内；后处理转换到 xyxy 后仍应 clip。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(hidden_dim, 2 * hidden_dim), nn.ReLU(),
                                     nn.Linear(2 * hidden_dim, hidden_dim))
    def forward(self, x):
        return self.network(x)

class EncoderLayer(nn.Module):
    def __init__(self, hidden_dim=16, num_heads=4):
        super().__init__()
        self.attention = ManualMultiHeadAttention(hidden_dim, num_heads)
        self.norm1, self.norm2 = nn.LayerNorm(hidden_dim), nn.LayerNorm(hidden_dim)
        self.ffn = FeedForward(hidden_dim)

    def forward(self, x, position, padding_mask):
        qk = x + position
        x = self.norm1(x + self.attention(qk, qk, x, padding_mask))
        return self.norm2(x + self.ffn(x))

class DecoderLayer(nn.Module):
    def __init__(self, hidden_dim=16, num_heads=4):
        super().__init__()
        self.self_attention = ManualMultiHeadAttention(hidden_dim, num_heads)
        self.cross_attention = ManualMultiHeadAttention(hidden_dim, num_heads)
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(hidden_dim) for _ in range(3))
        self.ffn = FeedForward(hidden_dim)

    def forward(self, target, query_position, memory, memory_position, memory_mask):
        q = target + query_position
        target = self.norm1(target + self.self_attention(q, q, target))
        target = self.norm2(target + self.cross_attention(target + query_position,
                                                           memory + memory_position, memory,
                                                           memory_mask))
        return self.norm3(target + self.ffn(target))

class BoxMLP(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim, 4))
    def forward(self, x):
        return self.layers(x).sigmoid()

class TinyDETR(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=16, num_heads=4,
                 num_queries=3, num_classes=2):
        super().__init__()
        self.in_channels, self.hidden_dim = int(in_channels), int(hidden_dim)
        self.num_heads = int(num_heads)
        self.num_queries, self.num_classes = int(num_queries), int(num_classes)
        self.backbone = TinyCNNBackbone(in_channels, hidden_dim)
        self.position = Sine2DPositionEncoding(hidden_dim)
        self.encoder = EncoderLayer(hidden_dim, num_heads)
        self.decoder = DecoderLayer(hidden_dim, num_heads)
        self.query_embedding = nn.Embedding(num_queries, hidden_dim)
        self.class_head = nn.Linear(hidden_dim, num_classes + 1)
        self.box_head = BoxMLP(hidden_dim)

    def forward(self, images, pixel_padding_mask=None):
        if images.ndim != 4 or images.shape[1] != self.in_channels:
            raise ValueError("TinyDETR image shape mismatch")
        batch, _, height, width = images.shape
        if pixel_padding_mask is None:
            pixel_padding_mask = torch.zeros(batch, height, width, dtype=torch.bool, device=images.device)
        if pixel_padding_mask.shape != (batch, height, width) or pixel_padding_mask.dtype != torch.bool:
            raise ValueError("pixel padding mask must be bool [B,H,W]")
        if pixel_padding_mask.all(dim=(1, 2)).any():
            raise ValueError("an image cannot be entirely padding")
        clean_images = images.masked_fill(pixel_padding_mask[:, None], 0.0)
        features = self.backbone(clean_images)
        feature_mask = self.backbone.propagate_padding_mask(pixel_padding_mask)
        if feature_mask.shape != (batch, *features.shape[-2:]):
            raise RuntimeError("mask propagation must match backbone feature shape")
        position_2d = self.position(feature_mask)
        memory = features.flatten(2).transpose(1, 2)
        memory_position = position_2d.flatten(1, 2)
        flat_mask = feature_mask.flatten(1)
        memory = self.encoder(memory, memory_position, flat_mask)
        query_position = self.query_embedding.weight.unsqueeze(0).expand(batch, -1, -1)
        target = torch.zeros_like(query_position)
        decoded = self.decoder(target, query_position, memory, memory_position, flat_mask)
        return {"pred_logits": self.class_head(decoded), "pred_boxes": self.box_head(decoded)}

detr_probe = TinyDETR().eval()
probe_images = torch.randn(2, 1, 16, 16, requires_grad=True)
probe_predictions = detr_probe(probe_images)
probe_predictions["pred_logits"].mean().backward()
assert probe_predictions["pred_logits"].shape == (2, 3, 3)
assert probe_predictions["pred_boxes"].shape == (2, 3, 4)
assert ((probe_predictions["pred_boxes"] > 0) & (probe_predictions["pred_boxes"] < 1)).all()
assert probe_images.grad is not None and float(probe_images.grad.norm()) > 0
assert detr_probe.query_embedding.weight.grad is not None

## 6. padding 不变性与感受野级 mask

若 mask 声明右侧四列是 padding，那么任意修改这些像素都不应改变输出。测试必须在 `eval()` 下进行，排除 BatchNorm 状态变化。输入先清零还不够：三层 `k=3,s=2,p=1` 卷积后，一个 feature token 只要其感受野碰到 padding 就不能作为有效 key。因此 mask 必须逐层使用同参数的 max-pool 保守传播，最终 feature map 是 `2×2`，不能用 adaptive pooling 猜边界。

全 padding 样本没有任何有效 key，attention softmax 无定义，接口应直接拒绝。第 7 列边界 oracle 专门区分真正的感受野传播与错误的 adaptive pooling。

In [ ]:
padding_model = TinyDETR().eval()
base_image = torch.randn(1, 1, 16, 16)
pixel_mask = torch.zeros(1, 16, 16, dtype=torch.bool)
pixel_mask[:, :, 12:] = True
changed_image = base_image.clone()
changed_image[:, :, :, 12:] = 999.0
with torch.no_grad():
    base_output = padding_model(base_image, pixel_mask)
    changed_output = padding_model(changed_image, pixel_mask)
assert torch.allclose(base_output["pred_logits"], changed_output["pred_logits"], atol=1e-6)
assert torch.allclose(base_output["pred_boxes"], changed_output["pred_boxes"], atol=1e-6)
try:
    padding_model(base_image, torch.ones_like(pixel_mask))
    raise AssertionError("all-padding image must fail")
except ValueError:
    pass

# 第 7 列同时落入最终两个横向 token 的卷积感受野；adaptive pooling 会漏掉右 token。
boundary_mask = torch.zeros(1, 16, 16, dtype=torch.bool)
boundary_mask[:, :, 7] = True
rf_mask = padding_model.backbone.propagate_padding_mask(boundary_mask)
adaptive_mask = F.adaptive_max_pool2d(boundary_mask.float().unsqueeze(1), (2, 2)).squeeze(1).bool()
assert rf_mask.shape == (1, 2, 2)
assert rf_mask[0, 0].tolist() == [True, True]
assert adaptive_mask[0, 0].tolist() == [True, False]
assert not torch.equal(rf_mask, adaptive_mask)


## 7. box 表示、IoU 与匹配代价

网络输出 normalized `cxcywh`，损失和评估常转换到 `xyxy`。两框 IoU 为

$$\operatorname{IoU}(a,b)=\frac{|a\cap b|}{|a|+|b|-|a\cap b|}.$$

匹配代价组合类别负对数概率、$L_1$ box 距离和 $1-\text{IoU}$。每一项尺度不同，权重属于训练配置，必须进入制品或实验记录。这里的穷举复杂度为 $P(Q,T)=Q!/(Q-T)!$，只适合 $Q,T$ 很小的数值 oracle。

In [ ]:
def validate_detection_target35(target, num_classes):
    if not isinstance(target, dict) or set(target) != {"labels", "boxes"}:
        raise ValueError("target must contain exactly labels and boxes")
    labels, boxes = target["labels"], target["boxes"]
    if labels.dtype != torch.long or labels.ndim != 1 or boxes.shape != (labels.numel(), 4):
        raise ValueError("invalid target tensor shape or dtype")
    if labels.numel() and ((labels < 0).any() or (labels >= num_classes).any()):
        raise ValueError("target labels must be real classes, never the no-object index")
    if not torch.isfinite(boxes).all():
        raise ValueError("target boxes must be finite")
    if boxes.numel():
        if (boxes[:, 2:] <= 0).any():
            raise ValueError("target width/height must be positive")
        xyxy = cxcywh_to_xyxy(boxes)
        if (xyxy < 0).any() or (xyxy > 1).any():
            raise ValueError("target boxes must stay inside normalized image bounds")

def cxcywh_to_xyxy(boxes):
    if boxes.shape[-1] != 4:
        raise ValueError("boxes must end in four coordinates")
    center, size = boxes[..., :2], boxes[..., 2:]
    return torch.cat([center - size / 2, center + size / 2], dim=-1)

def pairwise_iou(boxes1, boxes2):
    if boxes1.ndim != 2 or boxes2.ndim != 2:
        raise ValueError("pairwise_iou expects two matrices")
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    intersection = (bottom_right - top_left).clamp_min(0).prod(dim=-1)
    area1 = (boxes1[:, 2:] - boxes1[:, :2]).clamp_min(0).prod(dim=-1)
    area2 = (boxes2[:, 2:] - boxes2[:, :2]).clamp_min(0).prod(dim=-1)
    union = area1[:, None] + area2[None, :] - intersection
    return intersection / union.clamp_min(1e-8)

def optimal_bipartite_match(pred_logits, pred_boxes, target_labels, target_boxes,
                            class_weight=1.0, l1_weight=3.0, iou_weight=2.0):
    queries = pred_logits.shape[0]
    targets = target_labels.numel()
    if (pred_logits.ndim != 2 or pred_logits.shape[1] < 2 or
            pred_boxes.shape != (queries, 4) or target_boxes.shape != (targets, 4)):
        raise ValueError("matching tensor shape mismatch")
    validate_detection_target35({"labels": target_labels, "boxes": target_boxes},
                                pred_logits.shape[1] - 1)
    if not all(math.isfinite(value) and value >= 0
               for value in (class_weight, l1_weight, iou_weight)):
        raise ValueError("matching weights must be finite and non-negative")
    if targets == 0:
        empty = torch.empty(0, dtype=torch.long, device=pred_logits.device)
        return empty, empty
    if targets > queries:
        raise ValueError("targets exceed available object queries")
    class_cost = -pred_logits.log_softmax(-1)[:, target_labels]
    l1_cost = torch.cdist(pred_boxes, target_boxes, p=1)
    iou_cost = 1 - pairwise_iou(cxcywh_to_xyxy(pred_boxes), cxcywh_to_xyxy(target_boxes))
    cost = class_weight * class_cost + l1_weight * l1_cost + iou_weight * iou_cost
    best_assignment, best_value = None, float("inf")
    for assignment in permutations(range(queries), targets):
        value = float(sum(cost[assignment[t], t].detach() for t in range(targets)))
        if value < best_value:
            best_value, best_assignment = value, assignment
    return (torch.tensor(best_assignment, device=pred_logits.device),
            torch.arange(targets, device=pred_logits.device))

# 两个完全命中的 box 应得到单位 IoU；不相交得到 0。
iou_oracle = pairwise_iou(torch.tensor([[0., 0., 1., 1.], [0., 0., .2, .2]]),
                          torch.tensor([[0., 0., 1., 1.], [.8, .8, 1., 1.]]))
assert torch.allclose(iou_oracle.diag(), torch.tensor([1., 0.]))
match_logits = torch.tensor([[6., 0., -2.], [0., 6., -2.], [-2., -2., 6.]])
match_boxes = torch.tensor([[.2, .2, .2, .2], [.8, .8, .2, .2], [.5, .5, .1, .1]])
matched_q, matched_t = optimal_bipartite_match(match_logits, match_boxes,
                                                torch.tensor([1, 0]),
                                                torch.tensor([[.8, .8, .2, .2], [.2, .2, .2, .2]]))
assert matched_q.tolist() == [1, 0] and matched_t.tolist() == [0, 1]

## 8. set loss、no-object 与排列不变性

匹配后，所有 query 都进入 classification loss：匹配 query 使用真实类别，未匹配 query 使用 no-object。由于空类别数量通常很多，给 no-object 较小权重 `eos_coef`。只有匹配 query 计算 box $L_1$ 和 IoU loss；空目标时 box loss 应为可反传的精确零。

对预测 query 任意置换，总损失必须不变。这项 metamorphic test 能发现“按下标和 target 对齐”的伪 DETR 实现。

In [ ]:
def set_loss_single(pred_logits, pred_boxes, target, num_classes=2,
                    eos_coef=0.2, bbox_weight=3.0, iou_weight=2.0):
    if (pred_logits.ndim != 2 or pred_logits.shape[1] != num_classes + 1 or
            pred_boxes.shape != (pred_logits.shape[0], 4) or pred_logits.shape[0] < 1):
        raise ValueError("prediction shape mismatch")
    if not (torch.isfinite(pred_logits).all() and torch.isfinite(pred_boxes).all()):
        raise ValueError("predictions must be finite")
    if not math.isfinite(eos_coef) or eos_coef <= 0:
        raise ValueError("eos_coef must be finite and positive")
    validate_detection_target35(target, num_classes)
    labels, boxes = target["labels"], target["boxes"]
    matched_q, matched_t = optimal_bipartite_match(pred_logits, pred_boxes, labels, boxes)
    class_targets = torch.full((pred_logits.shape[0],), num_classes,
                               dtype=torch.long, device=pred_logits.device)
    if matched_q.numel():
        class_targets[matched_q] = labels[matched_t]
    per_query_ce = F.cross_entropy(pred_logits, class_targets, reduction="none")
    query_weights = torch.ones_like(per_query_ce)
    query_weights[class_targets == num_classes] = eos_coef
    # 固定 query 数作 denominator；不再被 F.cross_entropy(weight=..., mean) 的权重和抵消。
    class_loss = (per_query_ce * query_weights).sum() / pred_logits.shape[0]
    if matched_q.numel():
        selected_pred, selected_target = pred_boxes[matched_q], boxes[matched_t]
        l1_loss = F.l1_loss(selected_pred, selected_target)
        iou_matrix = pairwise_iou(cxcywh_to_xyxy(selected_pred), cxcywh_to_xyxy(selected_target))
        iou_loss = 1 - iou_matrix.diag().mean()
    else:
        l1_loss = pred_boxes.sum() * 0.0
        iou_loss = pred_boxes.sum() * 0.0
    total = class_loss + bbox_weight * l1_loss + iou_weight * iou_loss
    return total, {"class": class_loss, "l1": l1_loss, "iou": iou_loss}

def batch_set_loss(outputs, targets):
    losses = [set_loss_single(outputs["pred_logits"][i], outputs["pred_boxes"][i], target)
              for i, target in enumerate(targets)]
    return torch.stack([item[0] for item in losses]).mean()

random_logits = torch.randn(4, 3)
random_boxes = torch.rand(4, 4)
two_targets = {"labels": torch.tensor([0, 1]),
               "boxes": torch.tensor([[.2, .3, .2, .3], [.7, .6, .25, .2]])}
loss_original = set_loss_single(random_logits, random_boxes, two_targets)[0]
query_permutation = torch.tensor([2, 0, 3, 1])
loss_permuted = set_loss_single(random_logits[query_permutation],
                                random_boxes[query_permutation], two_targets)[0]
assert torch.allclose(loss_original, loss_permuted, atol=1e-6)

empty_target = {"labels": torch.empty(0, dtype=torch.long), "boxes": torch.empty(0, 4)}
empty_total, empty_parts = set_loss_single(random_logits, random_boxes, empty_target)
empty_low = set_loss_single(random_logits, random_boxes, empty_target, eos_coef=0.1)[0]
empty_high = set_loss_single(random_logits, random_boxes, empty_target, eos_coef=0.5)[0]
assert empty_total > 0 and empty_parts["class"] > 0
assert empty_parts["l1"].item() == 0.0 and empty_parts["iou"].item() == 0.0
assert torch.allclose(empty_high, 5.0 * empty_low, atol=1e-6)  # eos 权重对空图真实生效

invalid_targets = [
    {"labels": torch.tensor([2]), "boxes": torch.tensor([[.5, .5, .2, .2]])},
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[.5, .5, 0., .2]])},
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[.95, .5, .2, .2]])},
    {"labels": torch.tensor([0]), "boxes": torch.tensor([[float("nan"), .5, .2, .2]])},
]
for invalid_target in invalid_targets:
    try:
        set_loss_single(random_logits, random_boxes, invalid_target)
        raise AssertionError("invalid target must fail")
    except ValueError:
        pass

## 9. 离线合成检测数据

每张 $16\times16$ 图像含一个矩形：类别 0 是横向矩形，类别 1 是纵向矩形；中心位置与噪声变化。target 保持为每图一个字典，box 是相对尺寸的 `cxcywh`。

真实检测 batch 有不同图像大小和不同目标数，需要 pad image 并生成 pixel mask；本数据固定尺寸是为了隔离 matching 与 box 学习。前面的测试已经单独覆盖 padding 行为，后面的空目标测试覆盖零目标分支。

In [ ]:
def make_detection_batch(count, seed):
    generator = torch.Generator().manual_seed(seed)
    images, targets = torch.zeros(count, 1, 16, 16), []
    for index in range(count):
        label = index % 2
        box_width, box_height = ((6, 3) if label == 0 else (3, 6))
        # 三个受控位置在各 split 重复出现，噪声随 seed 改变；便于快速验证定位计算图。
        left = (2, 6, 9)[(index // 2) % 3]
        top = (2, 7, 9)[(index // 6) % 3]
        left = min(left, 15 - box_width)
        top = min(top, 15 - box_height)
        images[index, 0, top:top + box_height, left:left + box_width] = 1.0
        images[index] += 0.03 * torch.randn(images[index].shape, generator=generator)
        cx = (left + box_width / 2) / 16
        cy = (top + box_height / 2) / 16
        targets.append({"labels": torch.tensor([label], dtype=torch.long),
                        "boxes": torch.tensor([[cx, cy, box_width / 16, box_height / 16]],
                                              dtype=torch.float32)})
    return images, targets

train_images, train_targets = make_detection_batch(12, SEED + 1)
valid_images, valid_targets = make_detection_batch(12, SEED + 2)
test_images, test_targets = make_detection_batch(12, SEED + 3)
assert train_images.shape == (12, 1, 16, 16)
assert all(target["boxes"].shape == (1, 4) for target in train_targets)
assert all(((target["boxes"] >= 0) & (target["boxes"] <= 1)).all() for target in train_targets)
assert {int(target["labels"][0]) for target in train_targets} == {0, 1}

## 10. 受控训练与 validation checkpoint

训练同时更新 CNN、position-aware encoder/decoder、object queries 和两个 heads。每 10 步仅用 validation 选择 checkpoint；test 不参与选择。由于 matching 离散，匹配索引不求梯度，但选中的 logits/box loss 对网络保持可微。

第一步检查 backbone、query embedding、class head 和 box head 都有有限非零梯度。训练损失下降只说明实现可学习；数据极简，不能拿这里的数值同 COCO 指标比较。

In [ ]:
torch.manual_seed(SEED)
model35 = TinyDETR().to(DEVICE)
optimizer = torch.optim.Adam(model35.parameters(), lr=0.015)
history, best_validation, best_state = [], float("inf"), None

for step in range(41):
    model35.train()
    optimizer.zero_grad(set_to_none=True)
    predictions = model35(train_images)
    loss = batch_set_loss(predictions, train_targets)
    loss.backward()
    if step == 0:
        gradient_checks = {
            "backbone": model35.backbone.network[0].weight.grad.norm(),
            "queries": model35.query_embedding.weight.grad.norm(),
            "class_head": model35.class_head.weight.grad.norm(),
            "box_head": model35.box_head.layers[-1].weight.grad.norm(),
        }
    torch.nn.utils.clip_grad_norm_(model35.parameters(), 2.0)
    optimizer.step()
    history.append(float(loss.detach()))
    if step % 10 == 0:
        model35.eval()
        with torch.no_grad():
            validation_loss = float(batch_set_loss(model35(valid_images), valid_targets))
        if validation_loss < best_validation:
            best_validation, best_state = validation_loss, deepcopy(model35.state_dict())

assert best_state is not None
model35.load_state_dict(best_state)
assert all(torch.isfinite(value) and float(value) > 0 for value in gradient_checks.values())
assert sum(history[-10:]) / 10 < history[0] * 0.75
print({"train_loss_first_last": [history[0], history[-1]],
       "best_validation_set_loss": best_validation})

## 11. 推理：去掉 no-object、clip box，再评估

推理不能对包含 no-object 的全部类别直接 `argmax` 后当目标返回。先取 softmax，保留最佳真实类别及其分数，再按阈值过滤；box 从 cxcywh 转 xyxy 并 clip 到 `[0,1]`。同一 query 只给一个预测，DETR 通常不依赖 NMS。

本例报告每图最高分预测的类别准确率和 IoU，仅作为受控 smoke metric。真实检测必须实现按类别 AP、不同 IoU threshold 的 mAP、small/medium/large 分桶、空图 false positives 与置信度校准。

In [ ]:
@torch.no_grad()
def postprocess_detr(outputs, score_threshold=0.25):
    probabilities = outputs["pred_logits"].softmax(-1)
    real_scores, real_labels = probabilities[..., :-1].max(-1)
    no_object_scores = probabilities[..., -1]
    boxes_xyxy = cxcywh_to_xyxy(outputs["pred_boxes"]).clamp(0.0, 1.0)
    results = []
    for sample in range(probabilities.shape[0]):
        keep = (real_scores[sample] >= score_threshold) & (real_scores[sample] > no_object_scores[sample])
        results.append({"scores": real_scores[sample, keep],
                        "labels": real_labels[sample, keep],
                        "boxes": boxes_xyxy[sample, keep]})
    return results

model35.eval()
with torch.no_grad():
    test_outputs = model35(test_images)
test_results = postprocess_detr(test_outputs, score_threshold=0.20)
best_ious, correct_classes = [], []
for result, target in zip(test_results, test_targets):
    assert result["boxes"].ndim == 2 and result["boxes"].shape[-1] == 4
    assert ((result["boxes"] >= 0) & (result["boxes"] <= 1)).all()
    if len(result["scores"]) == 0:
        best_ious.append(0.0); correct_classes.append(0.0)
        continue
    best = int(result["scores"].argmax())
    target_xyxy = cxcywh_to_xyxy(target["boxes"])
    best_ious.append(float(pairwise_iou(result["boxes"][best:best+1], target_xyxy)[0, 0]))
    correct_classes.append(float(result["labels"][best] == target["labels"][0]))

mean_iou = sum(best_ious) / len(best_ious)
class_accuracy = sum(correct_classes) / len(correct_classes)
mean_train_box = torch.cat([target["boxes"] for target in train_targets]).mean(0, keepdim=True)
constant_box_iou = sum(
    float(pairwise_iou(cxcywh_to_xyxy(mean_train_box),
                       cxcywh_to_xyxy(target["boxes"]))[0, 0])
    for target in test_targets
) / len(test_targets)
print({"controlled_test_mean_best_iou": mean_iou,
       "controlled_test_class_accuracy": class_accuracy,
       "constant_box_baseline_iou": constant_box_iou,
       "detections_per_image": [len(item["scores"]) for item in test_results]})
assert mean_iou >= constant_box_iou + 0.03
assert class_accuracy >= 0.80

## 12. 检测制品：外部信任锚、数据快照与语义合同

“manifest 里放一个 hash”只能发现传输损坏，不能阻止攻击者整体替换模型后把内部 hash 全部重签。这里把信任边界改成两层：调用方 package 仍携带内部摘要，但 loader 还必须命中发布者侧只读登记表 `artifact_id/version -> expected bundle digest`。bundle 摘要对 canonical manifest 与逐 tensor 的 `key/dtype/shape/bytes` 一起做长度分隔哈希；登记表不来自 package。

检测语义也必须整体绑定：模型 config、类别顺序、`normalized_cxcywh`、CNN 感受野 mask 传播、score threshold、matching/loss 权重，以及 train/validation/test 的图像、label、box 与 split seed。加载器会重建本受控数据并复核 snapshot。生产中这个只读登记表应由签名发布元数据、透明日志或只读制品服务提供，而不是和模型放在同一个可替换目录。

In [ ]:
def canonical_json35(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False).encode("utf-8")

def _feed_digest35(hasher, payload):
    hasher.update(len(payload).to_bytes(8, "big"))
    hasher.update(payload)

def clone_state35(state_dict):
    if not hasattr(state_dict, "items"):
        raise ValueError("state_dict must be a mapping")
    cloned = {}
    for key, tensor in state_dict.items():
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("state entries must be string -> Tensor")
        cloned[key] = tensor.detach().cpu().contiguous().clone()
    return cloned

def _update_state_digest35(hasher, state_dict):
    if not isinstance(state_dict, dict) or not state_dict:
        raise ValueError("artifact state_dict must be a non-empty plain dict")
    for key in sorted(state_dict):
        tensor = state_dict[key]
        if not isinstance(key, str) or not isinstance(tensor, torch.Tensor):
            raise ValueError("invalid state entry")
        cpu = tensor.detach().cpu().contiguous()
        header = canonical_json35({"key": key, "dtype": str(cpu.dtype),
                                   "shape": list(cpu.shape)})
        raw = cpu.reshape(-1).view(torch.uint8).numpy().tobytes()
        _feed_digest35(hasher, header)
        _feed_digest35(hasher, raw)

def canonical_state_digest35(state_dict):
    hasher = sha256()
    _feed_digest35(hasher, b"canonical-state-dict-v1")
    _update_state_digest35(hasher, state_dict)
    return hasher.hexdigest()

def canonical_bundle_digest35(manifest, state_dict):
    hasher = sha256()
    _feed_digest35(hasher, b"canonical-model-bundle-v1")
    _feed_digest35(hasher, canonical_json35(manifest))
    _update_state_digest35(hasher, state_dict)
    return hasher.hexdigest()

def state_schema35(state_dict):
    return [{"key": key, "dtype": str(state_dict[key].dtype),
             "shape": list(state_dict[key].shape)} for key in sorted(state_dict)]

def detection_split_digest35(images, targets):
    tensors = {"images": images}
    for index, target in enumerate(targets):
        tensors[f"target/{index}/labels"] = target["labels"]
        tensors[f"target/{index}/boxes"] = target["boxes"]
    return canonical_state_digest35(clone_state35(tensors))

def expected_data_contract35():
    split_specs = {"train": (12, SEED + 1), "validation": (12, SEED + 2),
                   "test": (12, SEED + 3)}
    splits = {}
    for name, (count, seed) in split_specs.items():
        images, targets = make_detection_batch(count, seed)
        splits[name] = {"count": count, "seed": seed,
                        "images_targets_sha256": detection_split_digest35(images, targets)}
    return {"dataset_recipe": "controlled-rectangles-v1", "splits": splits,
            "target_schema": {"labels": "int64[N]", "boxes": "float32[N,4]"}}

EXPECTED_CONFIG35 = {"in_channels": 1, "hidden_dim": 16, "num_heads": 4,
                     "num_queries": 3, "num_classes": 2}
EXPECTED_CLASSES35 = ["horizontal", "vertical"]
EXPECTED_PREPROCESS35 = {
    "input_shape": [1, 16, 16], "dtype": "float32", "normalization": "none",
    "padding_mask_true_means": "padding",
    "padding_mask_downsample": "conv-receptive-field-max-pool-k3-s2-p1-x3/v1",
}
EXPECTED_MATCHING35 = {"algorithm": "exact-global-exhaustive-oracle",
                       "class_weight": 1.0, "l1_weight": 3.0, "iou_weight": 2.0}
EXPECTED_LOSS35 = {"classification": "weighted-ce-fixed-query-denominator",
                   "eos_coef": 0.2, "bbox_weight": 3.0, "iou_weight": 2.0}
ARTIFACT_ID35, ARTIFACT_VERSION35 = "vision.controlled-detr", "1.0.0"

def make_detr_artifact(model):
    if type(model) is not TinyDETR:
        raise ValueError("publisher only accepts the audited TinyDETR class")
    config = {"in_channels": model.in_channels, "hidden_dim": model.hidden_dim,
              "num_heads": model.num_heads, "num_queries": model.num_queries,
              "num_classes": model.num_classes}
    state = clone_state35(model.state_dict())
    manifest = {
        "schema_version": 2, "artifact_id": ARTIFACT_ID35,
        "artifact_version": ARTIFACT_VERSION35, "architecture": "TinyDETR",
        "config": config, "model_state_schema": state_schema35(state),
        "class_names": list(EXPECTED_CLASSES35),
        "preprocess": deepcopy(EXPECTED_PREPROCESS35),
        "box_format": "normalized_cxcywh", "score_threshold": 0.20,
        "matching": deepcopy(EXPECTED_MATCHING35), "loss": deepcopy(EXPECTED_LOSS35),
        "data_contract": expected_data_contract35(),
        "state_digest_sha256": canonical_state_digest35(state),
    }
    return {"manifest": manifest,
            "manifest_sha256": sha256(canonical_json35(manifest)).hexdigest(),
            "bundle_sha256": canonical_bundle_digest35(manifest, state),
            "state_dict": state}

def validate_detr_contract35(manifest, state_dict):
    required = {"schema_version", "artifact_id", "artifact_version", "architecture",
                "config", "model_state_schema", "class_names", "preprocess",
                "box_format", "score_threshold", "matching", "loss",
                "data_contract", "state_digest_sha256"}
    if set(manifest) != required or manifest["schema_version"] != 2:
        raise ValueError("manifest schema mismatch")
    if (manifest["artifact_id"], manifest["artifact_version"]) != (ARTIFACT_ID35, ARTIFACT_VERSION35):
        raise ValueError("artifact identity mismatch")
    if manifest["architecture"] != "TinyDETR" or manifest["config"] != EXPECTED_CONFIG35:
        raise ValueError("model config mismatch")
    names = manifest["class_names"]
    if (names != EXPECTED_CLASSES35 or len(names) != EXPECTED_CONFIG35["num_classes"] or
            len(set(names)) != len(names) or any(not isinstance(x, str) or not x.strip() for x in names)):
        raise ValueError("class mapping mismatch")
    if manifest["preprocess"] != EXPECTED_PREPROCESS35:
        raise ValueError("preprocess contract mismatch")
    if manifest["box_format"] != "normalized_cxcywh" or manifest["score_threshold"] != 0.20:
        raise ValueError("postprocess contract mismatch")
    if manifest["matching"] != EXPECTED_MATCHING35 or manifest["loss"] != EXPECTED_LOSS35:
        raise ValueError("matching/loss contract mismatch")
    if manifest["data_contract"] != expected_data_contract35():
        raise ValueError("image/target/split snapshot mismatch")
    expected_schema = state_schema35(TinyDETR(**EXPECTED_CONFIG35).state_dict())
    if manifest["model_state_schema"] != expected_schema or state_schema35(state_dict) != expected_schema:
        raise ValueError("model state schema mismatch")

def load_trusted_detr(artifact):
    if not isinstance(artifact, dict) or set(artifact) != {
            "manifest", "manifest_sha256", "bundle_sha256", "state_dict"}:
        raise ValueError("artifact package schema mismatch")
    manifest, state = artifact["manifest"], artifact["state_dict"]
    if not isinstance(manifest, dict):
        raise ValueError("manifest must be a dict")
    actual_bundle = canonical_bundle_digest35(manifest, state)
    identity = (manifest.get("artifact_id"), manifest.get("artifact_version"))
    expected_bundle = PUBLISHER_REGISTRY35.get(identity)
    if expected_bundle is None or actual_bundle != expected_bundle:
        raise ValueError("publisher registry rejected this bundle")
    if artifact["bundle_sha256"] != actual_bundle:
        raise ValueError("internal bundle digest mismatch")
    if sha256(canonical_json35(manifest)).hexdigest() != artifact["manifest_sha256"]:
        raise ValueError("manifest digest mismatch")
    if canonical_state_digest35(state) != manifest["state_digest_sha256"]:
        raise ValueError("canonical state digest mismatch")
    validate_detr_contract35(manifest, state)
    loaded = TinyDETR(**manifest["config"])
    loaded.load_state_dict(state, strict=True)
    return loaded.eval()

def resign_inside35(artifact):
    artifact["manifest"]["state_digest_sha256"] = canonical_state_digest35(artifact["state_dict"])
    artifact["manifest_sha256"] = sha256(canonical_json35(artifact["manifest"])).hexdigest()
    artifact["bundle_sha256"] = canonical_bundle_digest35(artifact["manifest"], artifact["state_dict"])
    return artifact

artifact35 = make_detr_artifact(model35)
PUBLISHER_REGISTRY35 = MappingProxyType({
    (ARTIFACT_ID35, ARTIFACT_VERSION35): artifact35["bundle_sha256"]
})
loaded35 = load_trusted_detr(artifact35)
with torch.no_grad():
    original = model35(test_images[:2])["pred_logits"]
    restored = loaded35(test_images[:2])["pred_logits"]
assert torch.equal(original, restored)

# 攻击者整体换成 4-query 模型，并同时篡改 box 语义、重签所有 package 内摘要。
# 内部摘要完全自洽，但发布者登记的 bundle digest 没变，因此仍必须 fail closed。
forged_model35 = TinyDETR(num_queries=4)
forged35 = deepcopy(artifact35)
forged35["state_dict"] = clone_state35(forged_model35.state_dict())
forged35["manifest"]["config"]["num_queries"] = 4
forged35["manifest"]["model_state_schema"] = state_schema35(forged35["state_dict"])
forged35["manifest"]["box_format"] = "absolute_xyxy"
resign_inside35(forged35)
assert forged35["bundle_sha256"] == canonical_bundle_digest35(forged35["manifest"], forged35["state_dict"])
try:
    load_trusted_detr(forged35)
    raise AssertionError("self-signed whole replacement must fail")
except ValueError:
    pass

try:
    PUBLISHER_REGISTRY35[(ARTIFACT_ID35, ARTIFACT_VERSION35)] = forged35["bundle_sha256"]
    raise AssertionError("publisher registry must be immutable")
except TypeError:
    pass

## 13. 失败模式、复杂度与生产差距

- **把 target 按下标对齐 query**：预测顺序改变时 loss 改变；排列不变测试应失败。
- **空图跳过 classification loss**：模型学不会 no-object，线上会产生大量假阳性。
- **只在 attention mask padding**：padding 像素可能已通过 CNN 污染有效 feature；输入先清零，并按每层卷积感受野把 mask 保守传播到 2×2 feature。
- **全 mask 后 softmax**：会产生 NaN；应在进入 attention 前拒绝没有有效像素的样本。
- **box 单位混用**：normalized cxcywh、absolute xyxy 必须由 manifest 绑定。
- **穷举匹配上生产**：复杂度随目标数阶乘增长；实际使用 Hungarian/Jonker–Volgenant 等经过测试的多项式算法。

attention 的时间/显存复杂度约为 encoder $O(S^2D)$、decoder cross-attention $O(QSD)$；高分辨率 feature 会主导成本。生产系统还需多尺度特征、小目标策略、数据增强、分布式训练、mixed precision、COCO 风格 mAP、拥挤/遮挡/空图/OOD 分桶、阈值校准、延迟与显存压测。

### 论文来源

- Carion et al., [*End-to-End Object Detection with Transformers*](https://arxiv.org/abs/2005.12872), ECCV 2020.
- Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), NeurIPS 2017（scaled dot-product attention）。
- Kuhn, [*The Hungarian Method for the Assignment Problem*](https://doi.org/10.1002/nav.3800020109), 1955（最优二分匹配背景）。

这里复现 DETR 的集合预测骨架与工程合同，不代表复现 COCO 训练配方或论文指标。